# Feature Engineering — Joint XGBoost + Block Optuna

Uses the **Optuna-tuned XGBoost** setup from `models copy.ipynb`:
- Hyperparameter search space: `build_xgb_classifier()`
- Saved baseline: `optuna_best_extended.json`
- Base feature blocks: `FEATURE_BLOCKS` from models copy

This notebook adds **new feature-engineering blocks** and runs:
1. Fixed-parameter ablation (leave-one-out + add-one-new-block)
2. Joint Optuna: **XGBoost params + all ablation block toggles**

Metric: 5-fold stratified CV log loss (primary), AUC ROC (secondary).

## 1. Imports

Helper modules (same folder as notebook):
- `model_copy_utils.py` — XGB tuning setup from models copy
- `feature_eng_lib.py` — ablation blocks + submission helpers

Run this cell first. Section 8 can run standalone after cells 1, 2, and 4 if `feature_eng_joint_best.json` exists.

In [ ]:
import json
import sys
from pathlib import Path

import optuna
import pandas as pd

# Ensure project root is on path when running from notebook
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model_copy_utils import (
    CV_FOLDS,
    FEATURE_BLOCKS,
    MODEL_BUILDERS,
    OPTUNA_BEST_PATH,
    OPTUNA_CV_FOLDS,
    OPTUNA_RANDOM_STATE,
    RANDOM_STATE,
    build_tuned_xgb,
    build_xgb_classifier,
    evaluate_xgb_cv,
    get_feature_sets,
    load_data,
    load_optuna_best,
    load_xgb_best,
)
from feature_eng_lib import (
    ALL_ABLATION_BLOCKS,
    BASE_ABLATION_BLOCKS,
    NEW_ABLATION_BLOCKS,
    OPTUNA_N_TRIALS,
    OPTUNA_STORAGE,
    OPTUNA_BEST_PATH as FEATURE_ENG_BEST_PATH,
    all_blocks_active,
    build_feature_cols_from_blocks,
    engineer_all_features,
    evaluate_block_config,
    load_feature_eng_best,
    make_feature_eng_submission,
    make_joint_optuna_objective,
    run_fixed_params_ablation,
    store_feature_eng_best,
    summarize_block_effects,
)

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 2. Load tuned XGBoost from models copy

Reference model: best XGB trial from `models copy.ipynb` (sections 10–12).

In [ ]:
xgb_best = load_xgb_best()
XGB_PARAMS = xgb_best["params"]
MODELS_COPY_FEATURE_COLS = xgb_best["feature_cols"]

tuned_xgb = MODEL_BUILDERS["xgboost"](XGB_PARAMS)

train_df = engineer_all_features(load_data("train.csv"))
y = train_df["default"]

drop_cols = ["client_id", "default"]
all_feature_cols = [c for c in train_df.columns if c not in drop_cols]
FEATURE_SETS = get_feature_sets(all_feature_cols)

models_copy_baseline = evaluate_xgb_cv(
    tuned_xgb, train_df[MODELS_COPY_FEATURE_COLS], y
)

print(f"Models copy tuned feature label: {xgb_best['feature_label']}")
print(f"Models copy tuned val log loss: {xgb_best['val_log_loss_mean']:.6f}")
print(f"Models copy feature count: {len(MODELS_COPY_FEATURE_COLS)}")
print(f"XGB params: {json.dumps(XGB_PARAMS, indent=2)}")
print(f"\nRe-eval baseline on current data: {models_copy_baseline['val_log_loss_mean']:.6f}")

## 3. Ablation block definitions

**Base blocks** (from `models copy.ipynb`):
`demographics`, `delay_engineered`, `pay_status`, `pay_amounts`, `bill_amounts`, `credit_util`, `bill_trends`, `models_copy_new_engineered`

**New feature_eng blocks**:
`delay_trends`, `pay_amt_stats`, `payment_change`, `util_stats`, `delay_util_interactions`

In [ ]:
print(f"Base ablation blocks ({len(BASE_ABLATION_BLOCKS)}):")
for name, cols in BASE_ABLATION_BLOCKS.items():
    print(f"  {name}: {len(cols)} cols")

print(f"\nNew ablation blocks ({len(NEW_ABLATION_BLOCKS)}):")
for name, cols in NEW_ABLATION_BLOCKS.items():
    print(f"  {name}: {cols}")

full_feature_cols = build_feature_cols_from_blocks(all_blocks_active())
print(f"\nAll blocks ON -> {len(full_feature_cols)} features")

## 4. Fixed-parameter ablation (models copy XGB params)

Uses tuned hyperparameters from section 2; only feature blocks vary.

- **Leave-one-out:** all blocks ON, remove one block at a time
- **Add-one-new:** base blocks only, add one new block at a time

Negative `delta_val_log_loss` = improvement.

In [ ]:
leave_one_out_report, add_one_new_report = run_fixed_params_ablation(
    train_df, y, XGB_PARAMS
)

print("Leave-one-out ablation (reference: all blocks ON):")
print(leave_one_out_report.to_string(index=False))

print("\nAdd-one-new-block ablation (reference: base blocks only):")
print(add_one_new_report.to_string(index=False))

## 5. Optuna setup — joint params + blocks

Search space per trial:
- **Hyperparameters:** same ranges as `build_xgb_classifier()` in models copy
- **Feature blocks:** on/off toggle for each block in `ALL_ABLATION_BLOCKS`

Uses a separate SQLite DB so it does not conflict with `models copy` studies.

In [ ]:
feature_eng_study = optuna.create_study(
    study_name="xgb_joint_params_and_blocks",
    storage=OPTUNA_STORAGE,
    load_if_exists=True,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=OPTUNA_RANDOM_STATE),
)

joint_objective = make_joint_optuna_objective(train_df, y)

print(f"Optuna storage: {OPTUNA_STORAGE}")
print(f"Study: {feature_eng_study.study_name}")
print(f"Planned trials: {OPTUNA_N_TRIALS}")
print(f"Block toggles: {len(ALL_ABLATION_BLOCKS)}")
print(f"XGB params tuned: n_estimators, max_depth, learning_rate, subsample,")
print(f"                  colsample_bytree, reg_lambda, reg_alpha, min_child_weight")

## 6. Run joint Optuna search (manual)

Run when ready — can take several minutes.

In [ ]:
feature_eng_study.optimize(
    joint_objective,
    n_trials=200,
    show_progress_bar=True,
)

best = feature_eng_study.best_trial
print(f"Best val log loss: {best.value:.6f}")
print(f"Best enabled blocks: {best.user_attrs['enabled_blocks']}")
print(f"Best XGB params: {best.user_attrs['xgb_params']}")

## 7. Block impact report (from joint Optuna trials)

For each block: mean val log loss when block is **ON** vs **OFF** across all completed trials.

- Negative `delta_on_minus_off` → block helps when enabled
- Compare `delta_vs_reference_when_on` to models copy baseline from section 2

In [ ]:
reference_val = models_copy_baseline["val_log_loss_mean"]
block_effects = summarize_block_effects(feature_eng_study, reference_val)

FEATURE_ENG_BEST = store_feature_eng_best(feature_eng_study, reference_val)

comparison = pd.DataFrame(
    [
        {
            "source": "models_copy_tuned_baseline",
            "val_log_loss_mean": reference_val,
            "val_roc_auc_mean": models_copy_baseline["val_roc_auc_mean"],
            "n_features": len(MODELS_COPY_FEATURE_COLS),
        },
        {
            "source": "feature_eng_joint_best",
            "val_log_loss_mean": FEATURE_ENG_BEST["best_val_log_loss_mean"],
            "val_roc_auc_mean": FEATURE_ENG_BEST["best_val_roc_auc_mean"],
            "n_features": FEATURE_ENG_BEST["n_features"],
        },
    ]
)

print(f"Saved {FEATURE_ENG_BEST_PATH}")
print("\nModels copy baseline vs feature_eng joint best:")
print(comparison.to_string(index=False))

print("\nPer-block effects (negative delta_on_minus_off => block helps):")
print(block_effects.to_string(index=False))

block_effects

## 8. Test submission (best joint combination)

Train tuned XGB on full train data with best feature cols + params; write `submission_feature_eng.csv`.

**Minimal run:** sections 1 → 2 → 4 → 8 (requires `feature_eng_joint_best.json` from section 7, or from a prior run).

In [ ]:
# Uses FEATURE_ENG_BEST from section 7 if available, else loads feature_eng_joint_best.json
if "FEATURE_ENG_BEST" not in globals() or not FEATURE_ENG_BEST:
    FEATURE_ENG_BEST = load_feature_eng_best(FEATURE_ENG_BEST_PATH)

test_df = engineer_all_features(load_data("test.csv"))

submission_feature_eng = make_feature_eng_submission(
    train_df,
    test_df,
    y,
    best_config=FEATURE_ENG_BEST,
    output_path="submission_feature_eng.csv",
)

submission_feature_eng.head()

## 9. Hard-error analysis (tuned XGB baseline)

Uses **feature_eng joint best** features + params (`feature_eng_joint_best.json`). Does not change sections 1–8 results.

- Fixed 5-fold `StratifiedKFold` splits (`random_state=42`)
- Confidently wrong: misclassified with confidence > 0.9
- Compare **confident false negatives** vs **correctly predicted defaults**
- Investigate unstable `bill_pct_change` features

In [ ]:
import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import feature_eng_lib
importlib.reload(feature_eng_lib)

from feature_eng_lib import (
    TARGETED_BLOCKS,
    assign_error_groups,
    compute_oof_with_folds,
    engineer_targeted_features,
    load_feature_eng_best,
    prediction_confidence,
    standardized_mean_differences,
    summarize_oof_metrics,
)

sns.set_theme(style="whitegrid")
CONFIDENCE_THRESHOLD = 0.9

if "FEATURE_ENG_BEST" not in globals() or not FEATURE_ENG_BEST:
    FEATURE_ENG_BEST = load_feature_eng_best(FEATURE_ENG_BEST_PATH)

BASELINE_FEATURE_COLS = FEATURE_ENG_BEST["best_feature_cols"]
JOINT_XGB_PARAMS = FEATURE_ENG_BEST["best_xgb_params"]

train_targeted_df = engineer_targeted_features(load_data("train.csv"))
y = train_targeted_df["default"].astype(int)

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
FOLDS = list(cv.split(train_targeted_df[BASELINE_FEATURE_COLS], y))

baseline_oof, baseline_fold_rows = compute_oof_with_folds(
    train_targeted_df[BASELINE_FEATURE_COLS],
    y,
    FOLDS,
    JOINT_XGB_PARAMS,
)
baseline_hard_error_metrics = summarize_oof_metrics(
    y,
    baseline_oof,
    baseline_fold_rows,
    confidence_threshold=CONFIDENCE_THRESHOLD,
)

print(f"Baseline features: {len(BASELINE_FEATURE_COLS)}")
print(f"Saved joint-best val log loss (section 7): {FEATURE_ENG_BEST['best_val_log_loss_mean']:.6f}")
print(f"Section 9 fold-OOF val log loss: {baseline_hard_error_metrics['val_log_loss_mean']:.6f}")
print(f"Confident FN count (conf > {CONFIDENCE_THRESHOLD}): {baseline_hard_error_metrics['confident_fn_count']}")

In [ ]:
analysis_df = train_targeted_df[["client_id", "default"] + BASELINE_FEATURE_COLS].copy()
analysis_df["oof_prob"] = baseline_oof
analysis_df["confidence"] = prediction_confidence(baseline_oof)
analysis_df["pred_class"] = (baseline_oof >= 0.5).astype(int)
analysis_df["error_group"] = assign_error_groups(
    y, baseline_oof, confidence_threshold=CONFIDENCE_THRESHOLD
)

group_counts = analysis_df["error_group"].value_counts()
print("Prediction groups (confidence > 0.9 for confident errors):")
print(group_counts.to_string())

error_summary = (
    analysis_df.groupby("error_group")
    .agg(
        count=("client_id", "count"),
        mean_confidence=("confidence", "mean"),
        default_rate=("default", "mean"),
    )
    .sort_values("count", ascending=False)
)
print("\nGroup summary:")
print(error_summary.round(4).to_string())

rank_cols = [
    "client_id", "default", "oof_prob", "pred_class", "confidence",
    "error_group", "PAY_0", "max_delay", "mean_util", "bill_pct_change_1_2",
]
rank_cols = [c for c in rank_cols if c in analysis_df.columns]

for group_name in ["confident_fn", "confident_fp"]:
    subset = analysis_df[analysis_df["error_group"] == group_name].nlargest(10, "confidence")
    print(f"\nTop 10 {group_name}:")
    print(subset[rank_cols].to_string(index=False))

In [ ]:
confident_fn_df = analysis_df[analysis_df["error_group"] == "confident_fn"]
correct_default_df = analysis_df[analysis_df["error_group"] == "correct_default"]

fn_vs_correct_shift = standardized_mean_differences(
    confident_fn_df,
    correct_default_df,
    BASELINE_FEATURE_COLS,
)
fn_shift_table = (
    pd.DataFrame(
        {
            "feature": fn_vs_correct_shift.index,
            "std_shift_vs_correct_default": fn_vs_correct_shift.values,
        }
    )
    .assign(abs_shift=lambda d: d["std_shift_vs_correct_default"].abs())
    .sort_values("abs_shift", ascending=False)
)

print(
    f"Confident FN ({len(confident_fn_df):,}) vs correctly predicted defaults "
    f"({len(correct_default_df):,})"
)
print("\nTop 20 standardized mean shifts:")
print(fn_shift_table.head(20).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 6))
plot_data = fn_shift_table.head(15).sort_values("std_shift_vs_correct_default")
colors = [
    "#d62728" if x > 0 else "#1f77b4"
    for x in plot_data["std_shift_vs_correct_default"]
]
ax.barh(plot_data["feature"], plot_data["std_shift_vs_correct_default"], color=colors)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel("Std shift (confident FN − correct default)")
ax.set_title("Top 15 feature shifts: confident FN vs correct defaults")
plt.tight_layout()
plt.show()

In [ ]:
raw_pct_cols = [
    c
    for c in train_targeted_df.columns
    if c.startswith("bill_pct_change_")
    and not c.endswith("_robust")
    and not c.endswith("_clip")
]

instability_rows = []
for raw_col in raw_pct_cols:
    suffix = raw_col.replace("bill_pct_change_", "")
    robust_col = f"bill_pct_change_{suffix}_robust"
    clip_col = f"bill_pct_change_{suffix}_clip"
    if robust_col not in train_targeted_df.columns or clip_col not in train_targeted_df.columns:
        continue

    raw_vals = train_targeted_df[raw_col]
    instability_rows.append(
        {
            "feature": raw_col,
            "in_baseline": raw_col in BASELINE_FEATURE_COLS,
            "max_abs_raw": float(raw_vals.abs().max()),
            "p99_abs_raw": float(raw_vals.abs().quantile(0.99)),
            "max_abs_robust": float(train_targeted_df[robust_col].abs().max()),
            "max_abs_clip": float(train_targeted_df[clip_col].abs().max()),
        }
    )

if instability_rows:
    instability_table = pd.DataFrame(instability_rows).sort_values(
        "max_abs_raw", ascending=False
    )
    print("Bill pct-change instability (raw vs robust vs clipped):")
    print(instability_table.to_string(index=False))

    example_col = instability_table.iloc[0]["feature"]
    suffix = example_col.replace("bill_pct_change_", "")
    example_robust = f"bill_pct_change_{suffix}_robust"
    example_clip = f"bill_pct_change_{suffix}_clip"

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(
        train_targeted_df[example_col].clip(-20, 20), bins=40, ax=axes[0], color="#d62728"
    )
    axes[0].set_title(f"{example_col} (clipped display ±20)")
    axes[0].set_xlabel("raw pct change")

    compare_df = pd.DataFrame(
        {
            "robust": train_targeted_df[example_robust],
            "clipped": train_targeted_df[example_clip],
        }
    ).melt(var_name="variant", value_name="value")
    sns.kdeplot(data=compare_df, x="value", hue="variant", ax=axes[1])
    axes[1].set_title(f"Robust vs clipped alternatives ({example_col})")
    plt.tight_layout()
    plt.show()
else:
    print("No raw bill_pct_change columns found in engineered dataset.")

## 10. Targeted feature engineering + sequential block test

Adds features aimed at **defaults that look superficially low-risk**, then tests blocks **sequentially** against the section 9 baseline using the same `FOLDS`.

**Blocks (in order):**
1. `robust_bill_pct` — denominators `abs(BILL_AMT)+1000`, clipped pct changes
2. `pay_amt_stats` — zero-payment count, pay mean/std/max
3. `payment_change` — recent vs old pay means, payment deterioration
4. `stabilized_pay_bill` — pay/bill ratios with stabilized denominators
5. `low_risk_interactions` — low-util × high-limit, limit/util, pay stress interactions

Keep a block only if OOF log loss improves. No threshold or class-weight tuning.

In [ ]:
importlib.reload(feature_eng_lib)
from feature_eng_lib import run_sequential_targeted_block_search, TARGETED_BLOCK_ORDER

print("Targeted blocks to test:")
for block_name in TARGETED_BLOCK_ORDER:
    print(f"  {block_name}: {TARGETED_BLOCKS[block_name]}")

comparison_df, best_feature_cols, enabled_blocks = run_sequential_targeted_block_search(
    train_targeted_df,
    y,
    BASELINE_FEATURE_COLS,
    JOINT_XGB_PARAMS,
    FOLDS,
    block_order=TARGETED_BLOCK_ORDER,
    confidence_threshold=CONFIDENCE_THRESHOLD,
)

display_cols = [
    "rank",
    "scenario",
    "block_added",
    "kept",
    "n_features",
    "val_log_loss_mean",
    "val_log_loss_std",
    "val_roc_auc_mean",
    "train_val_log_loss_gap",
    "confident_fn_count",
    "delta_confident_fn",
]
print("\nRanked comparison (baseline + each block attempt):")
print(comparison_df[display_cols].round(6).to_string(index=False))

print(f"\nBlocks kept: {enabled_blocks if enabled_blocks else '(none)'}")
print(f"Best feature count: {len(best_feature_cols)}")
print(f"Best val log loss: {comparison_df['val_log_loss_mean'].min():.6f}")

In [ ]:
added_cols = [c for c in best_feature_cols if c not in BASELINE_FEATURE_COLS]

best_feature_set = {
    "baseline_feature_cols": BASELINE_FEATURE_COLS,
    "enabled_targeted_blocks": enabled_blocks,
    "added_feature_cols": added_cols,
    "best_feature_cols": best_feature_cols,
    "xgb_params": JOINT_XGB_PARAMS,
    "confidence_threshold": CONFIDENCE_THRESHOLD,
    "baseline_confident_fn_count": int(baseline_hard_error_metrics["confident_fn_count"]),
}


print("Best feature set after sequential targeted block search:")
print(f"  Baseline features: {len(BASELINE_FEATURE_COLS)}")
print(f"  Added features: {len(added_cols)}")
print(f"  Total best features: {len(best_feature_cols)}")
print(f"  Enabled blocks: {enabled_blocks}")

if added_cols:
    print("\nNew features kept:")
    for col in added_cols:
        print(f"  - {col}")
else:
    print("\nNo targeted block improved OOF log loss; baseline feature set remains best.")

best_feature_set

## 11. Model comparison — XGBoost vs Neural Network vs Logistic Stack

Compare three approaches on the **feature_eng joint best** engineered dataset with fixed 5-fold CV:

1. **Tuned XGBoost** — saved `feature_eng_joint_best.json` params
2. **PyTorch MLP** — GPU-capable regressor (`resolve_device("auto"|"cuda"|"cpu")`) with scaled features + OOF CV (Optuna-tuned)
3. **Logistic stack** — Optuna-tuned LightGBM + CatBoost base models stacked via `run_global_search.py`

Reports OOF log loss / ROC-AUC, prediction correlations, and a ranked comparison table.

In [ ]:
import importlib

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import StratifiedKFold

import model_comparison_lib
importlib.reload(model_comparison_lib)

from feature_eng_lib import compute_oof_with_folds, load_feature_eng_best
from model_comparison_lib import (
    build_base_oof_predictions,
    compute_logistic_stack_oof,
    compute_nn_oof_cv,
    config_from_classical_trial,
    default_nn_params,
    describe_device,
    evaluate_model_oof,
    make_classical_objective,
    make_nn_optuna_objective,
    oof_correlation_matrix,
    optimize_logistic_stack,
    resolve_device,
    summarize_predictions,
    xgb_config_from_saved,
)
from model_copy_utils import build_tuned_xgb

COMPARE_OPTUNA_STORAGE = "sqlite:///optuna_feature_eng_model_compare.db"
CLASSICAL_OPTUNA_TRIALS = 40
NN_OPTUNA_TRIALS = 25
STACK_OPTUNA_TRIALS = 60
# PyTorch device: "auto" | "cuda" | "cuda:0" | "cpu"
NN_DEVICE = resolve_device("auto")

if "FEATURE_ENG_BEST" not in globals() or not FEATURE_ENG_BEST:
    FEATURE_ENG_BEST = load_feature_eng_best(FEATURE_ENG_BEST_PATH)

COMPARE_FEATURE_COLS = FEATURE_ENG_BEST["best_feature_cols"]
COMPARE_XGB_PARAMS = FEATURE_ENG_BEST["best_xgb_params"]

compare_df = engineer_all_features(load_data("train.csv"))
compare_y = compare_df["default"].astype(int)
compare_X = compare_df[COMPARE_FEATURE_COLS]

compare_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
COMPARE_FOLDS = list(compare_cv.split(compare_X, compare_y))

print(f"Features: {len(COMPARE_FEATURE_COLS)} | Rows: {len(compare_X):,}")
print(f"PyTorch NN device: {describe_device(NN_DEVICE)}")
print(f"Joint-best saved log loss: {FEATURE_ENG_BEST['best_val_log_loss_mean']:.6f}")

In [ ]:
# 1) Tuned XGBoost OOF (feature_eng joint best)
xgb_oof, xgb_metrics = evaluate_model_oof(
    build_tuned_xgb(COMPARE_XGB_PARAMS),
    compare_X,
    compare_y,
    COMPARE_FOLDS,
)
print("Tuned XGBoost OOF:")
print(
    f"  log loss={xgb_metrics['val_log_loss_mean']:.6f} "
    f"(std={xgb_metrics['val_log_loss_std']:.6f})"
)
print(
    f"  ROC-AUC={xgb_metrics['val_roc_auc_mean']:.6f} "
    f"(std={xgb_metrics['val_roc_auc_std']:.6f})"
)

# 2) Optuna-tune LightGBM + CatBoost on the same feature set
classical_studies = {}
classical_configs = []

for model_name in ["lightgbm", "catboost"]:
    study = optuna.create_study(
        study_name=f"feature_eng_compare_{model_name}",
        storage=COMPARE_OPTUNA_STORAGE,
        load_if_exists=True,
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    completed = sum(
        1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE
    )
    remaining = max(0, CLASSICAL_OPTUNA_TRIALS - completed)
    if remaining:
        study.optimize(
            make_classical_objective(
                model_name, compare_X, compare_y, COMPARE_FOLDS, seed=RANDOM_STATE
            ),
            n_trials=remaining,
            show_progress_bar=False,
        )
    classical_studies[model_name] = study
    classical_configs.append(
        config_from_classical_trial(model_name, study.best_trial, COMPARE_FEATURE_COLS)
    )
    print(
        f"{model_name}: trial {study.best_trial.number} "
        f"log loss={study.best_value:.6f} "
        f"auc={study.best_trial.user_attrs['val_roc_auc_mean']:.6f}"
    )

In [ ]:
# 3) Logistic stack over XGB + tuned classical base models
stack_base_configs = [
    xgb_config_from_saved(
        COMPARE_FEATURE_COLS,
        COMPARE_XGB_PARAMS,
        val_log_loss=xgb_metrics["val_log_loss_mean"],
        val_roc_auc=xgb_metrics["val_roc_auc_mean"],
    ),
    *classical_configs,
]

base_oof = build_base_oof_predictions(
    stack_base_configs,
    compare_df,
    compare_y,
    folds=CV_FOLDS,
    seed=RANDOM_STATE,
)
print("Stack base models:", list(base_oof))

stack_result = optimize_logistic_stack(
    base_oof,
    compare_y,
    folds=CV_FOLDS,
    seed=RANDOM_STATE,
    n_trials=STACK_OPTUNA_TRIALS,
    show_progress_bar=False,
)
stack_oof = compute_logistic_stack_oof(base_oof, compare_y, stack_result, seed=RANDOM_STATE)

print(
    f"Logistic stack: log loss={stack_result['val_log_loss_mean']:.6f}, "
    f"AUC={stack_result['val_roc_auc_mean']:.6f}, "
    f"input_mode={stack_result['input_mode']}, C={stack_result['params']['C']:.4f}"
)

In [ ]:
# 4) Neural network OOF (train_nn.py format) with Optuna tuning
nn_study = optuna.create_study(
    study_name="feature_eng_compare_nn",
    storage=COMPARE_OPTUNA_STORAGE,
    load_if_exists=True,
    direction="minimize",
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE + 7),
)
nn_completed = sum(
    1 for t in nn_study.trials if t.state == optuna.trial.TrialState.COMPLETE
)
nn_remaining = max(0, NN_OPTUNA_TRIALS - nn_completed)
if nn_remaining:
    nn_study.optimize(
        make_nn_optuna_objective(
            compare_X, compare_y, COMPARE_FOLDS, device=NN_DEVICE, random_state=RANDOM_STATE
        ),
        n_trials=nn_remaining,
        show_progress_bar=False,
    )

best_nn_params = nn_study.best_trial.user_attrs.get("nn_params", default_nn_params())
print(f"NN best trial {nn_study.best_trial.number}: log loss={nn_study.best_value:.6f}")
print(f"NN params: {best_nn_params}")

nn_oof = compute_nn_oof_cv(
    compare_X,
    compare_y,
    COMPARE_FOLDS,
    best_nn_params,
    device=NN_DEVICE,
    random_state=RANDOM_STATE,
)
nn_summary = summarize_predictions("neural_network", nn_oof, compare_y)
print(
    f"NN OOF: log loss={nn_summary['val_log_loss_mean']:.6f}, "
    f"AUC={nn_summary['val_roc_auc_mean']:.6f}"
)

In [ ]:
# 5) Comparison table + correlations
model_comparison = pd.DataFrame(
    [
        summarize_predictions(
            "tuned_xgboost",
            xgb_oof,
            compare_y,
            {
                "val_log_loss_std": xgb_metrics["val_log_loss_std"],
                "train_val_log_loss_gap": xgb_metrics["train_val_log_loss_gap"],
                "n_features": len(COMPARE_FEATURE_COLS),
            },
        ),
        summarize_predictions(
            "logistic_stack",
            stack_oof,
            compare_y,
            {
                "stack_input_mode": stack_result["input_mode"],
                "stack_C": stack_result["params"]["C"],
                "n_base_models": len(stack_result["labels"]),
            },
        ),
        summarize_predictions(
            "neural_network",
            nn_oof,
            compare_y,
            {
                "nn_loss": best_nn_params.get("loss_name"),
                "nn_hidden_dims": str(best_nn_params.get("hidden_dims")),
            },
        ),
    ]
).sort_values("val_log_loss_mean")

print("Model comparison (OOF, lower log loss is better):")
print(model_comparison.round(6).to_string(index=False))

oof_dict = {
    "tuned_xgboost": xgb_oof,
    "logistic_stack": stack_oof,
    "neural_network": nn_oof,
}
corr = oof_correlation_matrix(oof_dict)
print("\nOOF prediction correlations (Pearson):")
print(corr.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(corr, annot=True, fmt=".3f", cmap="coolwarm", vmin=0.5, vmax=1.0, ax=axes[0])
axes[0].set_title("OOF correlation matrix")

plot_corr = compare_df[["default"]].assign(**oof_dict).corr(numeric_only=True).loc[
    oof_dict.keys(), "default"
]
sns.barplot(x=plot_corr.values, y=plot_corr.index, ax=axes[1], color="#1f77b4")
axes[1].set_xlabel("Pearson correlation with default")
axes[1].set_title("Target correlation by model")
plt.tight_layout()
plt.show()

model_comparison